# Carnatify — raga feature extraction on Colab Pro (GPU) — **v3**

Runs the production-matched Demucs+pyin pipeline (`raga_v2_pipeline.py`) over the
shankarkrish/archive.org concert downloads, so the raga classifier can be retrained
on features that match live inference exactly.

**Why v3:** the v1 features were normalized by a median-F0 "tonic" that matched the
true Sa only 10% of the time (measured against Saraga annotations) — that rotated
every feature vector randomly and capped CV accuracy at 9%. v3 estimates the tonic
from the tambura drone in the *mixed* audio (essentia `TonicIndianArtMusic`, 87%
within ±50 cents) before Demucs strips it. Output goes to a fresh
`carnatify_raga_cache_v3/` folder; the old cache is obsolete.

**Before running:**
1. Runtime → Change runtime type → **GPU**.
2. The same `carnatify_concert_audio.zip` you already uploaded to Drive is reused —
   nothing to re-upload.
3. Run all cells. Features accumulate in `MyDrive/carnatify_raga_cache_v3/` — one
   `.npz` per track — so the run is **resumable across Colab sessions**.

**After it finishes:** download the `carnatify_raga_cache_v3` folder from Drive
(right-click → Download) and tell Claude — it handles extraction, retraining, and
evaluation locally.

Time estimate: similar to v1 (~45-90 s/track with 4 workers; tonic estimation adds
a few seconds per track). Safe to stop and re-run anytime.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip -q install demucs soundfile librosa essentia
!git clone -q https://github.com/ShyamRavidath/carnatify.git /content/carnatify
!unzip -q -n /content/drive/MyDrive/carnatify_concert_audio.zip -d /content/
!ls /content/concert_audio | head

In [ ]:
import sys
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

sys.path.insert(0, '/content/carnatify')
from raga_v2_pipeline import process_track

AUDIO_ROOT = Path('/content/concert_audio')
CACHE_DIR = Path('/content/drive/MyDrive/carnatify_raga_cache_v3')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
EXCLUDED_LABELS = {'Rāgamālika'}  # a form (garland of ragas), not a raga label

jobs = []
for raga_dir in sorted(AUDIO_ROOT.iterdir()):
    if not raga_dir.is_dir() or raga_dir.name in EXCLUDED_LABELS:
        continue
    for mp3 in sorted(raga_dir.glob('*.mp3')):
        # must match train_raga_v2_archive.py's naming so local + Colab caches merge
        track_id = f'archive__{raga_dir.name}__{mp3.stem}'
        jobs.append((track_id, raga_dir.name, str(mp3)))

already = {p.stem for p in CACHE_DIR.glob('*.npz')}
todo = [j for j in jobs if j[0] not in already]
print(f'{len(jobs)} tracks total, {len(already)} cached, {len(todo)} to process')

def run(job):
    track_id, raga, path = job
    try:
        return track_id, process_track(track_id, raga, path, CACHE_DIR)
    except Exception as exc:
        return track_id, {'status': f'error: {exc}'}

done = ok = 0
tonic_methods = {}
with ProcessPoolExecutor(max_workers=4) as pool:
    for track_id, result in map(lambda f: f.result(), as_completed([pool.submit(run, j) for j in todo])):
        done += 1
        status = (result or {}).get('status', 'skipped (too short / no pitch)')
        if status in ('ok', 'cached'):
            ok += 1
        tm = (result or {}).get('tonic_method')
        if tm:
            tonic_methods[tm] = tonic_methods.get(tm, 0) + 1
        if status.startswith('error') or done % 10 == 0:
            print(f'[{done}/{len(todo)}] {track_id}: {status}', flush=True)
print(f'\nfinished: {ok} ok of {done} processed; cache now has {len(list(CACHE_DIR.glob("*.npz")))} tracks')
print('tonic methods:', tonic_methods, '— should be almost all "essentia"; a large median_f0 count means essentia failed to install')